# Dense → MoE sparse upcycling (23M dense GPT → 8-expert top-2 MoE)

1. Train a dense GPT (d=512, 6 layers, 23M params) on FineWeb-Edu for the first part of a single LR schedule and checkpoint it.
2. **Upcycle**: every MLP is copied into E identical experts plus a small random router (top-k gates renormalised to sum 1), so the MoE starts with *exactly* the dense loss. AdamW moments are copied into each expert.
3. Continue training the MoE for the rest of the schedule, next to a dense control run from the same checkpoint on identical batches.

Runtime → Change runtime type → GPU. Uses bf16 on A100/L4, fp16 + GradScaler on T4.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| bf16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
%%writefile model.py
"""
20M-class GPT with three activation-memory strategies, all with matched parameter counts:

  - "baseline":  standard pre-norm transformer, autograd stores every activation.
  - "rev_euler": two-stream reversible network (Gomez et al. 2017). The coupled update
                 (a,b) -> (a + f(b), b + g(a + f(b))) is a semi-implicit *Euler* step of a
                 2D ODE; its inverse is exact, so only the final pair is stored.
  - "rev_mid":   single-stream *explicit midpoint* (leapfrog / Verlet) reversible stack:
                 x_{i+1} = x_{i-1} + f_i(x_i). Symmetric, 2nd-order; exact inverse
                 x_{i-1} = x_{i+1} - f_i(x_i) from adjacent activation pairs. Full-width
                 blocks (no channel halving).

All variants: d_model=512, 6 layers (blocks), 8 heads, MLP 4x, RoPE, tied embeddings,
vocab 8192 -> 18.87M non-embedding params + 4.19M embedding = 23.06M total.

Streams/residual are kept in fp32; matmuns run under bf16 autocast.
A fused chunked cross-entropy keeps logits memory O(chunk) in fwd and bwd.
"""

import math
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import custom_fwd, custom_bwd



# ------------------------------------------------------------------- autocast plumbing
# A reversible backward recomputes each block inside torch.autograd.grad. Under autocast
# the bf16 copy of every weight is CACHED, and because the forward ran under no_grad the
# cached copy carries no grad_fn -- the recompute then cannot reach the fp32 parameter and
# autograd.grad(..., allow_unused=True) silently returns None for every Linear weight.
# The model still trains (LayerNorm gains and the embedding are reached directly), so the
# failure is invisible in the loss curve. Re-entering autocast with cache_enabled=False
# inside the backward forces a fresh, differentiable cast.

def _amp_state():
    return (torch.is_autocast_enabled("cuda"), torch.get_autocast_dtype("cuda"))


def _amp_ctx(state):
    enabled, dtype = state
    return torch.autocast("cuda", dtype=dtype, enabled=enabled, cache_enabled=False)


def _require_all_grads(grads, params):
    if any(g is None for g in grads):
        missing = sum(g is None for g in grads)
        raise RuntimeError(
            f"reversible backward produced no gradient for {missing}/{len(params)} block "
            "parameters; the autocast weight cache has detached them from the graph")


# ----------------------------------------------------------------------------- blocks

def rope_cos_sin(T, head_dim, device, base=10000.0):
    half = head_dim // 2
    inv_freq = base ** (-torch.arange(half, device=device, dtype=torch.float32) / half)
    t = torch.arange(T, device=device, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)                      # (T, half)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, T, H, D). Rotate-half convention on the last dim.
    D = x.shape[-1]
    x1, x2 = x[..., : D // 2], x[..., D // 2:]
    c = cos[None, :, None, :]
    s = sin[None, :, None, :]
    return torch.cat([x1 * c - x2 * s, x2 * c + x1 * s], dim=-1)


class Attention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h = n_heads
        self.hd = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3 * self.h, self.hd).chunk(3, dim=2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))   # (B, H, T, hd)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).reshape(B, T, C)
        return self.proj(y)


class MLP(nn.Module):
    def __init__(self, d_model, mult=4):
        super().__init__()
        self.fc = nn.Linear(d_model, mult * d_model, bias=False)
        self.proj = nn.Linear(mult * d_model, d_model, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    """Standard pre-norm block (used by baseline and rev_mid)."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model, bias=False)
        self.attn = Attention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model, bias=False)
        self.mlp = MLP(d_model)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.ln1(x), cos, sin)
        x = x + self.mlp(self.ln2(x))
        return x


class RevBlock(nn.Module):
    """Two-stream reversible block: f = attention on stream b, g = MLP on stream a."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model, bias=False)
        self.attn = Attention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model, bias=False)
        self.mlp = MLP(d_model)

    def f(self, b, cos, sin):
        return self.attn(self.ln1(b), cos, sin)

    def g(self, a, cos, sin):
        return self.mlp(self.ln2(a))


# ------------------------------------------------------------- fused chunked cross-entropy

class FusedCE(torch.autograd.Function):
    """Chunked cross-entropy over flattened tokens; logits never stored for backward.
    forward: loss = mean CE. backward: dlogits = (softmax - onehot) * scale per chunk,
    accumulated into dh and dW with bf16 matmuls."""

    @staticmethod
    @custom_fwd(device_type="cuda")
    def forward(ctx, h, weight, targets, chunk=8192):
        # h: (N, d) fp32, weight: (V, d), targets: (N,) int64
        ctx.chunk = chunk
        ctx.save_for_backward(h, weight, targets)
        N = h.shape[0]
        with torch.no_grad():
            loss = h.new_zeros((), dtype=torch.float32)
            for i in range(0, N, chunk):
                hc, tc = h[i : i + chunk], targets[i : i + chunk]
                logits = F.linear(hc, weight).float()
                loss = loss + F.cross_entropy(logits, tc, reduction="sum")
        return loss / N

    @staticmethod
    @custom_bwd(device_type="cuda")
    def backward(ctx, grad_out):
        h, weight, targets = ctx.saved_tensors
        N, d = h.shape
        chunk = ctx.chunk
        dh = torch.zeros_like(h)
        dW = torch.zeros_like(weight)
        scale = grad_out / N
        for i in range(0, N, chunk):
            hc, tc = h[i : i + chunk], targets[i : i + chunk]
            logits = F.linear(hc, weight).float()
            p = torch.softmax(logits, dim=-1)
            p[torch.arange(tc.shape[0], device=h.device), tc] -= 1.0
            p = p * scale
            dW += p.T @ hc                       # TF32 matmul (enabled in train.py)
            dh[i : i + chunk] = p @ weight
        return dh, dW, None, None


def lm_loss(h, emb_weight, targets, chunk=8192):
    """h: (B, T, d) -> scalar CE loss."""
    B, T, d = h.shape
    return FusedCE.apply(h.reshape(B * T, d), emb_weight, targets.reshape(B * T), chunk)


# ------------------------------------------------------------------------- reversible stacks

class RevEulerFn(torch.autograd.Function):
    """Two-stream reversible stack. Stores only the final (a, b); backward reconstructs
    the chain top-down and recomputes each block with grad to produce exact gradients."""

    @staticmethod
    @custom_fwd(device_type="cuda")
    def forward(ctx, a0, b0, blocks, cos, sin, exact_seed=False):
        ctx.blocks = blocks
        ctx.cos, ctx.sin = cos, sin
        ctx.amp = _amp_state()
        ctx.exact_seed = exact_seed
        ctx.seed = (a0, b0) if exact_seed else None
        with torch.no_grad():
            a, b = a0, b0
            for blk in blocks:
                a = a + blk.f(b, cos, sin)
                b = b + blk.g(a, cos, sin)   # g sees the UPDATED a (sequential coupling -> invertible)
        ctx.save_for_backward(a, b)
        return a, b

    @staticmethod
    @custom_bwd(device_type="cuda")
    def backward(ctx, da, db):
        blocks, cos, sin = ctx.blocks, ctx.cos, ctx.sin
        a, b = ctx.saved_tensors
        for i, blk in list(enumerate(blocks))[::-1]:
            if ctx.exact_seed and i == 0:
                a_prev, b_prev = ctx.seed          # exact (a_0, b_0); no cancellation
            else:
                with _amp_ctx(ctx.amp), torch.no_grad():
                    b_prev = b - blk.g(a, cos, sin)
                    a_prev = a - blk.f(b_prev, cos, sin)
            a_prev = a_prev.detach().requires_grad_(True)
            b_prev = b_prev.detach().requires_grad_(True)
            params = [p for p in blk.parameters() if p.requires_grad]
            with _amp_ctx(ctx.amp), torch.enable_grad():
                a_new = a_prev + blk.f(b_prev, cos, sin)
                b_new = b_prev + blk.g(a_new, cos, sin)
            grads = torch.autograd.grad(
                (a_new, b_new), [a_prev, b_prev] + params,
                grad_outputs=(da, db), allow_unused=True,
            )
            _require_all_grads(grads[2:], params)
            da, db = grads[0], grads[1]
            for p, g in zip(params, grads[2:]):
                if g is not None:
                    p.grad = g if p.grad is None else p.grad + g
            a, b = a_prev.detach(), b_prev.detach()
        return da, db, None, None, None, None


class LeapfrogFn(torch.autograd.Function):
    """Single-stream explicit-midpoint (leapfrog) reversible stack:
    x_{i+1} = x_{i-1} + f_i(x_i), x_{-1} = 0. Stores only (x_{N-1}, x_N)."""

    @staticmethod
    @custom_fwd(device_type="cuda")
    def forward(ctx, x0, blocks, cos, sin, exact_seed=False):
        ctx.blocks = blocks
        ctx.cos, ctx.sin = cos, sin
        ctx.amp = _amp_state()
        ctx.exact_seed = exact_seed
        with torch.no_grad():
            xm1 = torch.zeros_like(x0)
            x = x0
            for blk in blocks:
                xm1, x = x, xm1 + blk(x, cos, sin)
        ctx.save_for_backward(*((xm1, x, x0) if exact_seed else (xm1, x)))
        return xm1, x

    @staticmethod
    @custom_bwd(device_type="cuda")
    def backward(ctx, dxm1_top, dx_top):
        blocks, cos, sin = ctx.blocks, ctx.cos, ctx.sin
        saved = ctx.saved_tensors            # (x_{N-1}, x_N[, x_0])
        xi, xi1 = saved[0], saved[1]
        x0_exact = saved[2] if ctx.exact_seed else None
        gxi, gxi1 = dxm1_top, dx_top         # their total grads from above
        for i, blk in list(enumerate(blocks))[::-1]:
            if x0_exact is not None and i == 0:
                # Recovering x_0 by subtraction is catastrophically ill-conditioned: it is
                # embedding-scale while x_1..x_N carry the accumulated residual. x_0 is an
                # input to this Function, so use it directly and skip the subtraction.
                xi = x0_exact
                xim1 = torch.zeros_like(xi)
            else:
                with _amp_ctx(ctx.amp), torch.no_grad():
                    xim1 = xi1 - blk(xi, cos, sin)   # x_{i-1} = x_{i+1} - f_i(x_i)
            xim1 = xim1.detach().requires_grad_(True)
            xi_leaf = xi.detach().requires_grad_(True)
            params = [p for p in blk.parameters() if p.requires_grad]
            with _amp_ctx(ctx.amp), torch.enable_grad():
                xi1_new = xim1 + blk(xi_leaf, cos, sin)
            grads = torch.autograd.grad(
                (xi1_new,), [xim1, xi_leaf] + params,
                grad_outputs=(gxi1,), allow_unused=True,
            )
            _require_all_grads(grads[2:], params)
            # grads[0]: identity path to x_{i-1}; grads[1]: path through f_i into x_i
            new_gxim1 = grads[0]
            new_gxi = gxi + grads[1]
            for p, g in zip(params, grads[2:]):
                if g is not None:
                    p.grad = g if p.grad is None else p.grad + g
            xi, xi1 = xim1.detach(), xi.detach()
            gxi, gxi1 = new_gxim1, new_gxi
        # loop ends with (x_{-1}, x_0): gradient w.r.t. x0 is gxi1; x_{-1} was a constant zero
        return gxi1, None, None, None, None


# --------------------------------------------------------------------------------- models

class GPTBaseline(nn.Module):
    def __init__(self, vocab, d_model=512, n_layers=6, n_heads=8, seq=512):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model, bias=False)
        self.seq = seq
        cos, sin = rope_cos_sin(seq, d_model // n_heads, "cpu")
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)
        # scale residual output projections for stable depth
        for blk in self.blocks:
            nn.init.normal_(blk.attn.proj.weight, std=0.02 / math.sqrt(2 * n_layers))
            nn.init.normal_(blk.mlp.proj.weight, std=0.02 / math.sqrt(2 * n_layers))

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None, ce_chunk=8192):
        cos = self.cos.to(idx.device)[: idx.shape[1]]
        sin = self.sin.to(idx.device)[: idx.shape[1]]
        x = self.emb(idx)
        for blk in self.blocks:
            x = blk(x, cos, sin)
        x = self.ln_f(x)
        if targets is None:
            return x
        return lm_loss(x, self.emb.weight, targets, ce_chunk)


class GPTRevEuler(nn.Module):
    """Two-stream reversible GPT. Each stream is d_model wide; params per block match baseline."""
    exact_seed = False

    def __init__(self, vocab, d_model=512, n_layers=6, n_heads=8, seq=512):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.blocks = nn.ModuleList([RevBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model, bias=False)
        self.seq = seq
        cos, sin = rope_cos_sin(seq, d_model // n_heads, "cpu")
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.apply(GPTBaseline._init)
        for blk in self.blocks:
            nn.init.normal_(blk.attn.proj.weight, std=0.02 / math.sqrt(2 * n_layers))
            nn.init.normal_(blk.mlp.proj.weight, std=0.02 / math.sqrt(2 * n_layers))

    def forward(self, idx, targets=None, ce_chunk=8192):
        cos = self.cos.to(idx.device)[: idx.shape[1]]
        sin = self.sin.to(idx.device)[: idx.shape[1]]
        e = self.emb(idx)
        a, b = RevEulerFn.apply(e, e, self.blocks, cos, sin, self.exact_seed)
        x = self.ln_f(a + b)
        if targets is None:
            return x
        return lm_loss(x, self.emb.weight, targets, ce_chunk)


class GPTRevMid(nn.Module):
    """Single-stream leapfrog (explicit midpoint) reversible GPT; blocks identical to baseline."""
    exact_seed = False

    def __init__(self, vocab, d_model=512, n_layers=6, n_heads=8, seq=512):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model, bias=False)
        self.seq = seq
        cos, sin = rope_cos_sin(seq, d_model // n_heads, "cpu")
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.apply(GPTBaseline._init)
        for blk in self.blocks:
            nn.init.normal_(blk.attn.proj.weight, std=0.02 / math.sqrt(2 * n_layers))
            nn.init.normal_(blk.mlp.proj.weight, std=0.02 / math.sqrt(2 * n_layers))

    def forward(self, idx, targets=None, ce_chunk=8192):
        cos = self.cos.to(idx.device)[: idx.shape[1]]
        sin = self.sin.to(idx.device)[: idx.shape[1]]
        e = self.emb(idx)
        x_prev, x = LeapfrogFn.apply(e, self.blocks, cos, sin, self.exact_seed)
        out = self.ln_f(x_prev + x)
        if targets is None:
            return out
        return lm_loss(out, self.emb.weight, targets, ce_chunk)


class GPTRevMidSeed(GPTRevMid):
    """rev_mid, but the backward takes x_0 from the saved input instead of reconstructing
    it. Costs one extra stored activation; removes the block-0 gradient corruption."""
    exact_seed = True


class GPTRevEulerSeed(GPTRevEuler):
    """rev_euler with the same exact-seed treatment."""
    exact_seed = True


ARCHS = {"baseline": GPTBaseline, "rev_euler": GPTRevEuler, "rev_mid": GPTRevMid,
         "rev_mid_seed": GPTRevMidSeed, "rev_euler_seed": GPTRevEulerSeed}


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    emb = model.emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


In [ ]:
%%writefile train.py
"""Trainer for the 20M-class reversible-vs-baseline experiments.

Examples:
  python train.py --arch baseline --batch 16 --tokens 50e6 --out runs/baseline_b16
  python train.py --arch rev_mid  --batch 16 --tokens 50e6 --out runs/revmid_b16
  python train.py --arch rev_mid  --batch 96 --tokens 50e6 --out runs/revmid_bmax --lr 4.2e-3
  python train.py --arch rev_mid  --probe-batch 128          # 3-step memory probe
"""

import argparse
import json
import math
import subprocess
import threading
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from model import ARCHS, count_params

ROOT = Path(__file__).parent
DATA = ROOT / "data"
VOCAB = 8192

AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
SCALER = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))


def get_batch(data, idxs, seq, device):
    xs = np.stack([data[i : i + seq] for i in idxs])
    ys = np.stack([data[i + 1 : i + 1 + seq] for i in idxs])
    return (torch.from_numpy(xs.astype(np.int64)).to(device, non_blocking=True),
            torch.from_numpy(ys.astype(np.int64)).to(device, non_blocking=True))


@torch.no_grad()
def evaluate(model, val_data, seq, device, max_tokens=2_000_000, batch=8):
    model.eval()
    n = len(val_data) - seq - 1
    rng = np.random.default_rng(7)
    total, count = 0.0, 0
    done = 0
    while done < max_tokens:
        b = min(batch, (max_tokens - done) // seq)
        if b <= 0:
            break
        idxs = rng.integers(0, n, size=b)
        x, y = get_batch(val_data, idxs, seq, device)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss = model(x, y)
        total += loss.item() * b * seq
        count += b * seq
        done += b * seq
    model.train()
    return total / count


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", required=True, choices=list(ARCHS))
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--seq", type=int, default=512)
    ap.add_argument("--tokens", type=float, default=50e6)
    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--warmup-frac", type=float, default=0.02)
    ap.add_argument("--seed", type=int, default=1234)
    ap.add_argument("--out", default=None)
    ap.add_argument("--log-every", type=int, default=50)
    ap.add_argument("--ce-chunk", type=int, default=8192)
    ap.add_argument("--probe-batch", type=int, default=0, help="run 3 steps at this batch, report memory, exit")
    ap.add_argument("--save-ckpt", action="store_true")
    ap.add_argument("--thermal-guard", action="store_true",
                    help="stop the run when the GPU enters the throttled regime "
                         "(SW thermal slowdown flag active, or >=87C)")
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    device = "cuda"
    out = Path(args.out) if args.out else ROOT / "runs" / f"{args.arch}_b{args.batch}"
    out.mkdir(parents=True, exist_ok=True)

    train_mm = np.memmap(DATA / "train.bin", dtype=np.uint16, mode="r")
    val_mm = np.memmap(DATA / "val.bin", dtype=np.uint16, mode="r")

    model = ARCHS[args.arch](VOCAB, seq=args.seq).to(device)
    pcount = count_params(model)
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        (no_decay if p.ndim < 2 else decay).append(p)
    opt = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}],
        lr=args.lr, betas=(0.9, 0.95), eps=1e-8, fused=True)

    steps = int(args.tokens // (args.batch * args.seq))
    warmup = max(30, int(steps * args.warmup_frac))

    if args.probe_batch:
        args.batch = args.probe_batch
        steps = 3

    print(f"arch={args.arch} params={pcount} batch={args.batch} seq={args.seq} steps={steps} lr={args.lr}", flush=True)

    g = torch.Generator().manual_seed(args.seed)
    n_train = len(train_mm) - args.seq - 1

    guard_stop = threading.Event()
    guard_done = threading.Event()
    guard_state = {"reason": None}
    guard_samples = []

    def _guard_smi():
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=temperature.gpu,clocks.sm,power.draw,"
             "clocks_event_reasons.sw_thermal_slowdown", "--format=csv,noheader,nounits"],
            capture_output=True, text=True).stdout.strip()
        p = [v.strip() for v in out.split(",")]
        return float(p[0]), float(p[1]), float(p[2]), p[3] == "Active"

    def thermal_watchdog():
        hot, t0 = 0, time.time()
        while not guard_done.is_set():
            try:
                temp, clk, pwr, sw = _guard_smi()
            except Exception:
                time.sleep(2.0)
                continue
            guard_samples.append({"t": round(time.time() - t0, 1), "temp": temp,
                                  "clk": clk, "pwr": pwr, "sw_thermal": sw})
            # the documented throttled regime: SW thermal governor engaged (its flag,
            # or >=87C) at ANY clock level; require 2 consecutive hits
            throttled = temp >= 87 or sw
            hot = hot + 1 if throttled else 0
            if hot >= 2:
                guard_state["reason"] = (f"throttled: {temp:.0f}C {clk:.0f}MHz "
                                         f"sw_thermal_slowdown={sw}")
                guard_stop.set()
                return
            time.sleep(2.0)

    if args.thermal_guard:
        threading.Thread(target=thermal_watchdog, daemon=True).start()

    curve, mem_peak_alloc, mem_peak_res = [], 0, 0
    t_start = time.time()
    t_steady_start, tokens_steady = None, 0
    ema = None
    steps_done = steps

    for step in range(steps):
        lr = args.lr * min((step + 1) / warmup, 1.0) if step < warmup else \
             args.lr * (0.1 + 0.45 * (1 + math.cos(math.pi * (step - warmup) / (steps - warmup))))
        for group in opt.param_groups:
            group["lr"] = lr

        idxs = torch.randint(0, n_train, (args.batch,), generator=g).numpy()
        x, y = get_batch(train_mm, idxs, args.seq, device)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss = model(x, y, ce_chunk=args.ce_chunk)
        if SCALER.is_enabled():
            SCALER.scale(loss).backward()
            SCALER.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            SCALER.step(opt)
            SCALER.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        opt.zero_grad(set_to_none=True)

        lv = loss.item()
        ema = lv if ema is None else 0.95 * ema + 0.05 * lv
        curve.append((step, lv, lr))

        if step == 5:  # start steady-state timing after warm-up of allocator
            torch.cuda.synchronize()
            t_steady_start = time.time()
            tokens_steady = 0
        if step >= 5:
            tokens_steady += args.batch * args.seq

        mem_peak_alloc = max(mem_peak_alloc, torch.cuda.max_memory_allocated())
        mem_peak_res = max(mem_peak_res, torch.cuda.max_memory_reserved())

        if (step + 1) % args.log_every == 0 or step == steps - 1:
            el = time.time() - t_start
            tps = tokens_steady / max(time.time() - t_steady_start, 1e-9) if step >= 5 else 0
            print(f"step {step+1}/{steps} loss {lv:.4f} ema {ema:.4f} lr {lr:.2e} "
                  f"tok/s {tps/1e3:.1f}k elapsed {el:.0f}s", flush=True)

        if args.thermal_guard and guard_stop.is_set():
            steps_done = step + 1
            print(f"THERMAL GUARD: stopping at step {step+1}/{steps}: {guard_state['reason']}",
                  flush=True)
            break

    guard_done.set()
    wall = time.time() - t_start
    tps_steady = tokens_steady / max(time.time() - t_steady_start, 1e-9) if t_steady_start else 0.0
    stopped = args.thermal_guard and guard_state["reason"] is not None

    if stopped:
        val_loss, val_ppl = None, None
    else:
        val_loss = evaluate(model, val_mm, args.seq, device)
        val_ppl = math.exp(val_loss)
    metrics = {
        "arch": args.arch, "batch": args.batch, "seq": args.seq,
        "tokens_planned": steps * args.batch * args.seq, "steps_planned": steps,
        "steps_done": steps_done, "tokens_done": steps_done * args.batch * args.seq,
        "tokens": steps_done * args.batch * args.seq, "steps": steps_done,
        "lr": args.lr, "params": pcount,
        "final_train_loss_ema": ema,
        "final_val_loss": val_loss, "val_ppl": val_ppl,
        "tokens_per_s": tps_steady,
        "peak_mem_allocated_mb": mem_peak_alloc / 2**20,
        "peak_mem_reserved_mb": mem_peak_res / 2**20,
        "wall_s": wall, "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__, "seed": args.seed,
        "stopped_reason": guard_state["reason"],
        "guard_samples_n": len(guard_samples),
    }
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    np.save(out / "curve.npy", np.array(curve))
    if args.thermal_guard:
        (out / "guard.json").write_text(json.dumps(
            {"reason": guard_state["reason"], "samples": guard_samples}, indent=1))
    if args.save_ckpt:
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": steps}, out / "ckpt.pt")
    print(json.dumps(metrics, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile moe.py
"""
Sparse upcycling: turn a trained dense GPTBaseline into a top-k Mixture-of-Experts GPT.

Every block's MLP (fc: d->4d, proj: 4d->d) is copied into E identical experts and a
small random router is added (a zero router would tie every token onto experts 0/1). With top-k gates renormalised to sum to 1, identical
experts make the MoE layer compute *exactly* the dense MLP at conversion time, so the
upcycled model starts from the dense model's loss. Routing then differs per token, the
experts receive different gradients, and they diverge / specialise as training continues.

Router logits are computed in fp32. A Switch-style load-balancing loss
  aux = E * sum_e f_e * P_e   (f_e = fraction of routed slots, P_e = mean router prob)
is stored on the model (model.aux_loss) so evaluate() keeps returning pure CE.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

from model import GPTBaseline, MLP


class MoEMLP(nn.Module):
    def __init__(self, d_model, n_experts=8, top_k=2, mult=4):
        super().__init__()
        self.E, self.k = n_experts, top_k
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.w_fc = nn.Parameter(torch.empty(n_experts, mult * d_model, d_model))
        self.w_proj = nn.Parameter(torch.empty(n_experts, d_model, mult * d_model))
        self.aux_loss = None
        self.last_load = None            # fraction of routed slots per expert (detached)

    @classmethod
    def from_dense(cls, mlp: MLP, n_experts, top_k, router_std=0.02):
        d = mlp.fc.weight.shape[1]
        m = cls(d, n_experts, top_k, mult=mlp.fc.weight.shape[0] // d).to(mlp.fc.weight.device)
        with torch.no_grad():
            m.w_fc.copy_(mlp.fc.weight.unsqueeze(0).expand_as(m.w_fc))
            m.w_proj.copy_(mlp.proj.weight.unsqueeze(0).expand_as(m.w_proj))
            if router_std > 0:
                nn.init.normal_(m.router.weight, std=router_std)
            else:
                m.router.weight.zero_()
        return m

    def forward(self, x):
        B, T, d = x.shape
        xf = x.reshape(-1, d)
        N = xf.shape[0]
        with torch.autocast("cuda", enabled=False):
            logits = F.linear(xf.float(), self.router.weight.float())
            probs = logits.softmax(-1)                                   # (N, E)
        top_p, top_i = probs.topk(self.k, dim=-1)                        # (N, k)
        gates = top_p / top_p.sum(-1, keepdim=True)

        flat_e = top_i.reshape(-1)                                       # (N*k,)
        counts = torch.bincount(flat_e, minlength=self.E)
        load = counts.float() / flat_e.numel()
        self.aux_loss = self.E * (load * probs.mean(0)).sum()
        self.last_load = load.detach()

        order = flat_e.argsort()
        tok = order // self.k                                            # source token per slot
        g = gates.reshape(-1)[order]
        out = torch.zeros(N, d, device=x.device, dtype=torch.float32)
        start = 0
        for e, c in enumerate(counts.tolist()):
            if c == 0:
                continue
            idx = tok[start : start + c]
            h = F.linear(F.gelu(F.linear(xf[idx], self.w_fc[e])), self.w_proj[e])
            out.index_add_(0, idx, h.float() * g[start : start + c, None])
            start += c
        return out.view(B, T, d).to(x.dtype)


class GPTMoE(GPTBaseline):
    """GPTBaseline whose block MLPs are MoEMLPs. forward() returns CE; aux on self.aux_loss."""

    def forward(self, idx, targets=None, ce_chunk=8192):
        out = super().forward(idx, targets, ce_chunk)
        self.aux_loss = sum(b.mlp.aux_loss for b in self.blocks) / len(self.blocks)
        return out

    def expert_load(self):
        return torch.stack([b.mlp.last_load for b in self.blocks])        # (L, E)

    @torch.no_grad()
    def expert_divergence(self):
        """Per layer: mean_e ||W_e - mean(W)|| / ||mean(W)|| over fc+proj (0 at upcycling)."""
        res = []
        for b in self.blocks:
            r = []
            for W in (b.mlp.w_fc, b.mlp.w_proj):
                mu = W.mean(0, keepdim=True)
                r.append(((W - mu).flatten(1).norm(dim=1) / mu.norm()).mean().item())
            res.append(sum(r) / 2)
        return res


def upcycle(dense: GPTBaseline, n_experts=8, top_k=2, router_std=0.02) -> GPTMoE:
    """Returns a new GPTMoE sharing no storage with `dense`."""
    import copy
    moe = copy.deepcopy(dense)
    moe.__class__ = GPTMoE
    for blk in moe.blocks:
        blk.mlp = MoEMLP.from_dense(blk.mlp, n_experts, top_k, router_std)
    moe.aux_loss = None
    return moe


In [ ]:
%%writefile upcycle.py
"""Dense -> MoE sparse-upcycling experiment.

One LR schedule (warmup + cosine to 0.1x) spans the whole --total-tokens budget.

  phase 1  dense GPTBaseline trained from scratch up to --stop-tokens, checkpointed
           (model, AdamW state, batch-sampler RNG, schedule position).
  phase 2  resume from that checkpoint for the rest of the schedule, either
             --mode dense : control, the same dense model keeps training
             --mode moe   : every MLP upcycled to E experts / top-k (AdamW moments copied
                            into each expert, fresh router), then trained
           Both arms see exactly the same batches and learning rates.

  python upcycle.py dense --stop-tokens 20e6 --out runs/up_dense20M
  python upcycle.py resume --init runs/up_dense20M/ckpt.pt --mode dense --out runs/up_dense_cont
  python upcycle.py resume --init runs/up_dense20M/ckpt.pt --mode moe --experts 8 --topk 2 --out runs/up_moe8x2
"""

import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import torch

from model import GPTBaseline, count_params
from moe import GPTMoE, upcycle
from train import DATA, VOCAB, evaluate, get_batch

AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
SCALER = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))


def lr_at(step, steps, warmup, peak):
    if step < warmup:
        return peak * (step + 1) / warmup
    return peak * (0.1 + 0.45 * (1 + math.cos(math.pi * (step - warmup) / (steps - warmup))))


def param_groups(model):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        (no_decay if p.ndim < 2 else decay).append((name, p))
    return decay, no_decay


def make_opt(model, lr):
    decay, no_decay = param_groups(model)
    opt = torch.optim.AdamW(
        [{"params": [p for _, p in decay], "weight_decay": 0.1},
         {"params": [p for _, p in no_decay], "weight_decay": 0.0}],
        lr=lr, betas=(0.9, 0.95), eps=1e-8, fused=True)
    return opt, [n for n, _ in decay] + [n for n, _ in no_decay]


def transfer_opt_state(opt, moe, dense_names, dense_opt_sd, n_experts):
    """Seed the MoE optimizer with the dense AdamW moments. Expert tensors get the dense
    MLP's moments replicated E times; the router starts with no state."""
    src = {n: dense_opt_sd["state"][i] for i, n in enumerate(dense_names) if i in dense_opt_sd["state"]}
    copied, fresh = 0, []
    for name, p in moe.named_parameters():
        s_name = name.replace("mlp.w_fc", "mlp.fc.weight").replace("mlp.w_proj", "mlp.proj.weight")
        if s_name not in src:
            fresh.append(name)
            continue
        s = src[s_name]
        st = {}
        for k, v in s.items():
            if torch.is_tensor(v) and v.ndim > 0 and v.shape != p.shape:
                v = v.unsqueeze(0).expand(n_experts, *v.shape)
            st[k] = v.clone().contiguous() if torch.is_tensor(v) else v
        opt.state[p] = st
        copied += 1
    return copied, fresh


def train_segment(model, opt, train_mm, val_mm, g, args, start, end, steps, warmup, log):
    is_moe = isinstance(model, GPTMoE)
    n_train = len(train_mm) - args.seq - 1
    curve, evals, ema = [], [], None
    t0, tok_t0 = time.time(), None
    for step in range(start, end):
        lr = lr_at(step, steps, warmup, args.lr)
        for grp in opt.param_groups:
            grp["lr"] = lr
        idxs = torch.randint(0, n_train, (args.batch,), generator=g).numpy()
        x, y = get_batch(train_mm, idxs, args.seq, "cuda")
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            ce = model(x, y)
        aux = model.aux_loss if is_moe else None
        loss = ce + args.aux_coef * aux if is_moe else ce
        if SCALER.is_enabled():
            SCALER.scale(loss).backward()
            SCALER.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            SCALER.step(opt)
            SCALER.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        opt.zero_grad(set_to_none=True)

        cv = ce.item()
        ema = cv if ema is None else 0.95 * ema + 0.05 * cv
        curve.append((step, cv, aux.item() if is_moe else 0.0, lr))
        if step == start + 5:
            torch.cuda.synchronize()
            tok_t0 = time.time()
        if (step + 1) % args.log_every == 0 or step == end - 1:
            tps = (step - start - 5) * args.batch * args.seq / (time.time() - tok_t0) if tok_t0 else 0
            msg = (f"step {step+1}/{steps} ce {cv:.4f} ema {ema:.4f} lr {lr:.2e} "
                   f"tok/s {tps/1e3:.1f}k elapsed {time.time()-t0:.0f}s")
            if is_moe:
                ld = model.expert_load()
                msg += f" aux {aux.item():.3f} load[min {ld.min().item():.3f} max {ld.max().item():.3f}]"
            log(msg)
        if (step + 1) % args.eval_every == 0 or step == end - 1:
            v = evaluate(model, val_mm, args.seq, "cuda", max_tokens=args.eval_tokens)
            evals.append((step + 1, v))
            log(f"  eval step {step+1} tokens {(step+1)*args.batch*args.seq/1e6:.1f}M val {v:.4f}")
    wall = time.time() - t0
    tps = (end - start - 5) * args.batch * args.seq / (time.time() - tok_t0) if tok_t0 else 0
    return curve, evals, ema, wall, tps


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("phase", choices=["dense", "resume"])
    ap.add_argument("--init", default=None)
    ap.add_argument("--mode", choices=["dense", "moe"], default="dense")
    ap.add_argument("--experts", type=int, default=8)
    ap.add_argument("--topk", type=int, default=2)
    ap.add_argument("--router-std", type=float, default=0.02)
    ap.add_argument("--aux-coef", type=float, default=0.01)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--seq", type=int, default=512)
    ap.add_argument("--total-tokens", type=float, default=50e6)
    ap.add_argument("--stop-tokens", type=float, default=20e6)
    ap.add_argument("--lr", type=float, default=7.5e-4)
    ap.add_argument("--warmup-frac", type=float, default=0.02)
    ap.add_argument("--seed", type=int, default=1234)
    ap.add_argument("--log-every", type=int, default=100)
    ap.add_argument("--eval-every", type=int, default=250)
    ap.add_argument("--eval-tokens", type=int, default=500_000)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    logf = open(out / "stdout.log", "a")

    def log(m):
        print(m, flush=True)
        logf.write(m + "\n")
        logf.flush()

    train_mm = np.memmap(DATA / "train.bin", dtype=np.uint16, mode="r")
    val_mm = np.memmap(DATA / "val.bin", dtype=np.uint16, mode="r")
    steps = int(args.total_tokens // (args.batch * args.seq))
    warmup = max(30, int(steps * args.warmup_frac))
    info = {}

    if args.phase == "dense":
        torch.manual_seed(args.seed)
        model = GPTBaseline(VOCAB, seq=args.seq).cuda()
        opt, names = make_opt(model, args.lr)
        g = torch.Generator().manual_seed(args.seed)
        start, end = 0, int(args.stop_tokens // (args.batch * args.seq))
        arm = "dense_phase1"
    else:
        ck = torch.load(args.init, map_location="cuda", weights_only=False)
        torch.manual_seed(args.seed + 1)
        dense = GPTBaseline(VOCAB, seq=args.seq).cuda()
        dense.load_state_dict(ck["model"])
        g = torch.Generator()
        g.set_state(ck["gen_state"].cpu())
        start, end = ck["step"], steps
        assert ck["steps_total"] == steps, "resume must use the same --total-tokens/--batch"
        v_dense = evaluate(dense, val_mm, args.seq, "cuda", max_tokens=args.eval_tokens)
        if args.mode == "dense":
            model = dense
            opt, names = make_opt(model, args.lr)
            opt.load_state_dict(ck["opt"])
            arm = "dense_continue"
        else:
            model = upcycle(dense, args.experts, args.topk, args.router_std)
            del dense
            opt, names = make_opt(model, args.lr)
            copied, fresh = transfer_opt_state(opt, model, ck["opt_names"], ck["opt"], args.experts)
            log(f"optimizer state copied for {copied} tensors; fresh: {fresh}")
            arm = f"moe_{args.experts}x_top{args.topk}"
        v0 = evaluate(model, val_mm, args.seq, "cuda", max_tokens=args.eval_tokens)
        info = {"val_at_resume_dense_ckpt": v_dense, "val_at_resume_this_model": v0}
        log(f"resume at step {start}: dense ckpt val {v_dense:.5f} | {arm} val {v0:.5f} "
            f"(delta {v0 - v_dense:+.2e})")

    pc = count_params(model)
    if isinstance(model, GPTMoE):
        per_expert = model.blocks[0].mlp.w_fc[0].numel() + model.blocks[0].mlp.w_proj[0].numel()
        inactive = len(model.blocks) * (args.experts - args.topk) * per_expert
        pc["active_per_token"] = pc["total"] - inactive
    log(f"arm={arm} params={pc} steps {start}->{end} of {steps} lr={args.lr}")

    curve, evals, ema, wall, tps = train_segment(
        model, opt, train_mm, val_mm, g, args, start, end, steps, warmup, log)
    if info:
        evals = [(start, info["val_at_resume_this_model"])] + evals

    val_full = evaluate(model, val_mm, args.seq, "cuda")          # 2M-token val, as train.py
    metrics = {
        "arm": arm, "params": pc, "batch": args.batch, "seq": args.seq, "lr": args.lr,
        "start_step": start, "end_step": end, "steps_total": steps,
        "tokens_start": start * args.batch * args.seq, "tokens_end": end * args.batch * args.seq,
        "final_train_ce_ema": ema, "final_val_loss_2M": val_full, "val_ppl": math.exp(val_full),
        "tokens_per_s": tps, "wall_s": wall,
        "peak_mem_allocated_mb": torch.cuda.max_memory_allocated() / 2**20,
        "gpu": torch.cuda.get_device_name(0), **info,
    }
    if isinstance(model, GPTMoE):
        metrics.update(experts=args.experts, topk=args.topk, aux_coef=args.aux_coef,
                       router_std=args.router_std,
                       expert_divergence_per_layer=model.expert_divergence(),
                       final_expert_load=model.expert_load().tolist())
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    np.save(out / "curve.npy", np.array(curve))
    np.save(out / "evals.npy", np.array(evals))
    if args.phase == "dense":
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "opt_names": names,
                    "gen_state": g.get_state(), "step": end, "steps_total": steps}, out / "ckpt.pt")
    log(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile data_prep.py
"""Prepare data: stream a FineWeb-Edu parquet shard, train an 8k BPE tokenizer,
write ~57M tokens as uint16 binaries (train.bin / val.bin)."""

import io
import sys
import time
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

ROOT = Path(__file__).parent
DATA = ROOT / "data"
PARQUET = DATA / "fineweb-edu-00000.parquet"
VOCAB = 8192
TRAIN_TOKENS = 52_000_000
VAL_TOKENS = 5_000_000
TOTAL_NEEDED = TRAIN_TOKENS + VAL_TOKENS


def main():
    t0 = time.time()
    pf = pq.ParquetFile(PARQUET)
    print(f"parquet: {pf.metadata.num_rows} rows, {pf.metadata.num_row_groups} row groups")

    # ---- collect a text sample for tokenizer training (~25MB)
    sample, sample_chars = [], 0
    texts_iter = (batch.column("text").to_pylist()
                  for batch in pf.iter_batches(batch_size=2048, columns=["text"]))
    for batch_texts in texts_iter:
        for t in batch_texts:
            if len(t) >= 300:
                sample.append(t)
                sample_chars += len(t)
        if sample_chars > 25_000_000:
            break
    print(f"tokenizer sample: {len(sample)} docs, {sample_chars/1e6:.1f} MB")

    tok = Tokenizer(models.BPE(unk_token=None))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB,
        special_tokens=["<|endoftext|>"],
        show_progress=False,
    )
    tok.train_from_iterator(sample, trainer)
    eot = tok.token_to_id("<|endoftext|>")
    assert eot == 0, f"EOT id is {eot}, expected 0"
    tok.save(str(DATA / "bpe_8192.json"))
    print(f"tokenizer trained: vocab {tok.get_vocab_size()} ({time.time()-t0:.0f}s)")

    del sample

    # ---- tokenize the corpus into a uint16 buffer
    buf = np.empty(TOTAL_NEEDED + 1_000_000, dtype=np.uint16)
    n = 0
    n_docs = 0
    for batch_texts in texts_iter:
        texts = [t for t in batch_texts if len(t) >= 300]
        if not texts:
            continue
        encs = tok.encode_batch(texts, add_special_tokens=False)
        for e in encs:
            ids = e.ids
            if n + len(ids) + 1 > len(buf):
                break
            buf[n : n + len(ids)] = ids
            n += len(ids)
            buf[n] = eot
            n += 1
            n_docs += 1
        if n % 10_000_000 < 40_000:
            print(f"  {n/1e6:.1f}M tokens ({time.time()-t0:.0f}s)", flush=True)
        if n >= TOTAL_NEEDED:
            break

    print(f"tokenized {n/1e6:.1f}M tokens from {n_docs} docs ({time.time()-t0:.0f}s)")
    train = buf[:TRAIN_TOKENS]
    val = buf[TRAIN_TOKENS : TRAIN_TOKENS + VAL_TOKENS]
    train.tofile(DATA / "train.bin")
    val.tofile(DATA / "val.bin")
    print(f"train.bin {TRAIN_TOKENS/1e6:.0f}M tokens, val.bin {VAL_TOKENS/1e6:.0f}M tokens")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile plot_upcycle.py
"""Plot the dense -> MoE upcycling runs: curves_upcycle.png."""

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

R = Path(__file__).parent / "runs"
TOK = 16 * 512 / 1e6                           # M tokens per step
DENSE, MOE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"


def smooth(y, a=0.98):
    out, s = np.empty_like(y), y[0]
    for i, v in enumerate(y):
        s = a * s + (1 - a) * v
        out[i] = s
    return out


def load(name):
    d = R / name
    return np.load(d / "curve.npy"), np.load(d / "evals.npy"), json.loads((d / "metrics.json").read_text())


c1, e1, m1 = load("up_dense20M")
cd, ed, md = load("up_dense_cont")
cm, em, mm = load("up_moe8x2")
conv = m1["tokens_end"] / 1e6
end = md["tokens_end"] / 1e6
MOE_LABEL = f"MoE {mm['experts']}x top-{mm['topk']}"

plt.rcParams.update({"font.size": 10, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False})
fig, axs = plt.subplots(1, 3, figsize=(16, 4.8))

# (a) training CE over the whole schedule
ax = axs[0]
dense_full = np.concatenate([c1, cd])
ax.plot(dense_full[:, 0] * TOK, smooth(dense_full[:, 1]), color=DENSE, lw=2, label="dense (23M)")
# MoE curve carries the dense EMA state across the conversion so the smoothing is continuous
moe_full = np.concatenate([c1, cm])
s = smooth(moe_full[:, 1])[len(c1):]
ax.plot(cm[:, 0] * TOK, s, color=MOE, lw=2, label=MOE_LABEL + " (upcycled)")
ax.axvline(conv, color=MUTED, lw=1, ls="--")
ax.text(conv, 6.3, " upcycled here", color=MUTED, va="top")
ax.set_ylim(3.4, 6.5)
ax.set_title("Training cross-entropy (EMA)", loc="left", color=INK)

# (b) validation loss after conversion
ax = axs[1]
ax.plot(e1[:, 0] * TOK, e1[:, 1], color=DENSE, lw=2, marker="o", ms=3)
ax.plot(ed[:, 0] * TOK, ed[:, 1], color=DENSE, lw=2, marker="o", ms=3, label="dense")
ax.plot(em[:, 0] * TOK, em[:, 1], color=MOE, lw=2, marker="o", ms=3, label=MOE_LABEL)
ax.axvline(conv, color=MUTED, lw=1, ls="--")
ax.set_xlim(conv - 0.3 * conv, end * 1.01)
lo = min(ed[:, 1].min(), em[:, 1].min())
ax.set_ylim(lo - 0.05, e1[e1[:, 0] * TOK >= 0.7 * conv, 1].max() + 0.05)
for e, col in ((ed, DENSE), (em, MOE)):
    ax.annotate(f"{e[-1, 1]:.3f}", (e[-1, 0] * TOK, e[-1, 1]), xytext=(4, 0),
                textcoords="offset points", color=INK, va="center")
ax.set_title("Validation loss (0.5M-token slice)", loc="left", color=INK)

# (c) MoE minus dense on identical eval points
ax = axs[2]
steps = np.intersect1d(ed[:, 0], em[:, 0])
gap = np.array([em[em[:, 0] == s, 1][0] - ed[ed[:, 0] == s, 1][0] for s in steps])
ax.axhline(0, color=MUTED, lw=1)
ax.plot(steps * TOK, gap, color=MOE, lw=2, marker="o", ms=3)
ax.annotate(f"{gap[-1]:+.3f}", (steps[-1] * TOK, gap[-1]), xytext=(4, 0),
            textcoords="offset points", color=INK, va="center")
ax.set_title("Val loss: MoE minus dense (below 0 = MoE better)", loc="left", color=INK)

for ax in axs:
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_xlabel("tokens seen (M)")
axs[0].set_ylabel("loss (nats/token)")
axs[0].legend(frameon=False)
axs[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(Path(__file__).parent / "curves_upcycle.png", dpi=130)
print("wrote curves_upcycle.png")


## Data
Downloads one FineWeb-Edu shard (~2.1 GB) and tokenizes 57M tokens with an 8k BPE (a few minutes).

In [ ]:

!pip -q install tokenizers huggingface_hub

import os, shutil
from huggingface_hub import hf_hub_download

os.makedirs("data", exist_ok=True)
dest = "data/fineweb-edu-00000.parquet"
if not os.path.exists("data/train.bin"):
    if not os.path.exists(dest):
        p = hf_hub_download("HuggingFaceFW/fineweb-edu", "sample/10BT/000_00000.parquet",
                            repo_type="dataset")
        try:                      # symlink to avoid a second 2.1 GB on disk
            os.symlink(p, dest)
        except OSError:
            shutil.copy(p, dest)
    print(f"{dest}: {os.path.getsize(dest)/1e9:.2f} GB")


In [ ]:
import os
if not os.path.exists('data/train.bin'):
    !python data_prep.py
!ls -la data

In [ ]:

# ---- experiment knobs -------------------------------------------------------------
TOTAL_TOKENS = 50e6   # one LR schedule over the whole budget
STOP_TOKENS  = 20e6   # dense phase ends and the MoE is created here
EXPERTS, TOPK = 8, 2
LR, BATCH = 7.5e-4, 16

# quick look instead (about 10 min on a T4): TOTAL_TOKENS, STOP_TOKENS = 10e6, 4e6
common = f"--total-tokens {TOTAL_TOKENS} --stop-tokens {STOP_TOKENS} --lr {LR} --batch {BATCH}"
print(common)


## Sanity check: the conversion does not change the model's outputs

In [ ]:

# Upcycling must be function-preserving: identical experts + renormalised top-k gates
# reproduce the dense MLP exactly, whatever the router says.
import torch
from model import GPTBaseline
from moe import upcycle

torch.manual_seed(0)
dense = GPTBaseline(8192, seq=512).cuda().eval()
moe = upcycle(dense, EXPERTS, TOPK).eval()
x = torch.randint(0, 8192, (2, 512), device="cuda")
with torch.no_grad():
    d, m = dense(x), moe(x)
print(f"max |dense - moe| = {(d - m).abs().max().item():.2e}")
n = lambda mod: sum(p.numel() for p in mod.parameters())
print(f"params: dense {n(dense)/1e6:.1f}M -> MoE {n(moe)/1e6:.1f}M")
del dense, moe


## Phase 1: dense training (saves `runs/up_dense20M/ckpt.pt`)

In [ ]:
!python upcycle.py dense {common} --out runs/up_dense20M --log-every 200

## Phase 2a: upcycle to MoE and keep training
The first log line compares the val loss of the dense checkpoint and the freshly upcycled MoE: they should match to ~1e-6.

In [ ]:
!python upcycle.py resume --init runs/up_dense20M/ckpt.pt --mode moe --experts {EXPERTS} --topk {TOPK} {common} --out runs/up_moe8x2 --log-every 200

## Phase 2b: dense control from the same checkpoint (same batches, same LR)

In [ ]:
!python upcycle.py resume --init runs/up_dense20M/ckpt.pt --mode dense {common} --out runs/up_dense_cont --log-every 200

## Results

In [ ]:

import json
print(f"{'arm':20s} {'tokens':>12s} {'val@resume':>11s} {'final val (2M)':>15s} {'ppl':>7s} {'tok/s':>8s}")
for r in ["up_dense20M", "up_dense_cont", "up_moe8x2"]:
    m = json.load(open(f"runs/{r}/metrics.json"))
    vr = m.get("val_at_resume_this_model")
    print(f"{m['arm']:20s} {m['tokens_start']/1e6:5.1f}-{m['tokens_end']/1e6:4.1f}M "
          f"{(f'{vr:.4f}' if vr else '-'):>11s} {m['final_val_loss_2M']:15.4f} "
          f"{m['val_ppl']:7.1f} {m['tokens_per_s']/1e3:7.1f}k")
m = json.load(open("runs/up_moe8x2/metrics.json"))
print("\nexpert divergence per layer (0 = still identical copies):",
      [round(v, 3) for v in m["expert_divergence_per_layer"]])
print("final expert load (share of routed slots, ideal", round(1 / m["experts"], 3), "):")
for i, row in enumerate(m["final_expert_load"]):
    print(f"  layer {i}: " + " ".join(f"{v:.3f}" for v in row))


In [ ]:
!python plot_upcycle.py
from IPython.display import Image
Image('curves_upcycle.png')

In [ ]:
# optional: download results (checkpoint excluded)
!zip -qr upcycle_results.zip runs/up_* curves_upcycle.png -x '*.pt'
from google.colab import files
files.download('upcycle_results.zip')